# MemoryVerse: Live ColQwen2-v1.0 DocVQA Per-Query Evaluation
**Hardware Target:** Nvidia Tesla T4 GPU (Kaggle Cloud)
**Benchmark Split:** `vidore/docvqa_test_subsampled` ($N=500$ queries, $N=500$ document pages)
**Goal:** Generate genuine per-query integer target ranks, top-10 document ID rankings, and 10,000-resample bootstrap confidence intervals directly from live `vidore/colqwen2-v1.0` weights on Tesla T4.


In [1]:
# CELL 1: Environment Setup & Dependencies
import os, sys, subprocess
from pathlib import Path

print("=== Setting up Environment & Dependencies ===", flush=True)
working_dir = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("./working")
working_dir.mkdir(parents=True, exist_ok=True)

os.environ["HF_HOME"] = "/tmp/hf_cache"
os.environ["TRANSFORMERS_CACHE"] = "/tmp/hf_cache"
os.environ["TORCH_HOME"] = "/tmp/hf_cache"

subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "torchao"], check=False)
subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "colpali-engine>=0.3.0",
    "transformers>=4.45.0",
    "accelerate>=0.34.0",
    "datasets>=2.19.0",
    "tabulate", "scipy"
], check=False)
print("Dependencies installed successfully!", flush=True)


=== Setting up Environment & Dependencies ===
Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0
Dependencies installed successfully!


In [2]:
# CELL 2: GPU Hardware Diagnostics
import torch

print("=== GPU Hardware Diagnostics ===", flush=True)
print(f"PyTorch Version: {torch.__version__}")
cuda_avail = torch.cuda.is_available()
print(f"CUDA Available:  {cuda_avail}")
device = torch.device("cuda:0" if cuda_avail else "cpu")
if cuda_avail:
    print(f"Device Name:     {torch.cuda.get_device_name(0)}")
    print(f"VRAM Total:      {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")
    print(f"BF16 Supported:  {torch.cuda.is_bf16_supported()}")
dtype = torch.bfloat16 if cuda_avail and torch.cuda.is_bf16_supported() else torch.float16
print(f"Selected Torch Precision: {dtype}", flush=True)


=== GPU Hardware Diagnostics ===
PyTorch Version: 2.11.0+cu130
CUDA Available:  True
Device Name:     Tesla T4
VRAM Total:      14.56 GB
BF16 Supported:  True
Selected Torch Precision: torch.bfloat16


In [3]:
# CELL 3: Load Official ViDoRe DocVQA Test Split
from datasets import load_dataset
from PIL import Image

print("=== Loading vidore/docvqa_test_subsampled from Hugging Face ===", flush=True)
ds = load_dataset("vidore/docvqa_test_subsampled", split="test")
print(f"Loaded {len(ds)} evaluation items from ViDoRe DocVQA!", flush=True)

queries = [item["query"] for item in ds]
images = [item["image"].convert("RGB") for item in ds]
image_filenames = [item.get("image_filename", f"doc_{i}") for i, item in enumerate(ds)]
doc_ids = [str(item.get("docId", image_filenames[i])) for i, item in enumerate(ds)]
question_ids = [str(item.get("questionId", f"q_{i}")) for i, item in enumerate(ds)]

print(f"Sample 0 Query:     {queries[0]}")
print(f"Sample 0 Filename:  {image_filenames[0]}")
print(f"Sample 0 Doc ID:    {doc_ids[0]}")
print(f"Total Unique Images: {len(images)}", flush=True)


=== Loading vidore/docvqa_test_subsampled from Hugging Face ===
Loaded 500 evaluation items from ViDoRe DocVQA!
Sample 0 Query:     What is the dividend payout in 2012?
Sample 0 Filename:  rnbx0223
Sample 0 Doc ID:    4720
Total Unique Images: 500


In [4]:
# CELL 4: Load ColQwen2-v1.0 Model and Processor
from colpali_engine.models import ColQwen2, ColQwen2Processor

MODEL_ID = "vidore/colqwen2-v1.0"
print(f"=== Loading {MODEL_ID} on {device} ({dtype}) ===", flush=True)
processor = ColQwen2Processor.from_pretrained(MODEL_ID)
model = ColQwen2.from_pretrained(MODEL_ID, torch_dtype=dtype, device_map="cuda:0").eval()
print("Model loaded successfully!", flush=True)


=== Loading vidore/colqwen2-v1.0 on cuda:0 (torch.bfloat16) ===
Model loaded successfully!


In [5]:
# CELL 5: Live GPU Encoding of 500 Document Page Images
import time

print(f"=== Encoding {len(images)} document images on GPU ===", flush=True)
t_start_img = time.time()
batch_size_img = 4
doc_embeddings = []

for b_idx in range(0, len(images), batch_size_img):
    b_imgs = images[b_idx : b_idx + batch_size_img]
    batch_inputs = processor.process_images(b_imgs).to(device)
    with torch.no_grad():
        b_embs = model(**batch_inputs)
        # Store on CPU/GPU as tensor list
        for emb in b_embs:
            doc_embeddings.append(emb.to(dtype).cpu())
    if (b_idx // batch_size_img) % 10 == 0 or b_idx + batch_size_img >= len(images):
        elapsed = time.time() - t_start_img
        print(f"  Encoded {min(b_idx + batch_size_img, len(images))}/{len(images)} images in {elapsed:.1f}s...", flush=True)

img_elapsed = time.time() - t_start_img
print(f"Completed image encoding of {len(images)} document pages in {img_elapsed:.1f}s!", flush=True)

# Save document embeddings
import torch
from pathlib import Path
working_dir = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("./working")
working_dir.mkdir(parents=True, exist_ok=True)
torch.save(doc_embeddings, working_dir / "colqwen2_docvqa_doc_embeddings.pt")
print(f"Saved doc embeddings tensor list ({len(doc_embeddings)} docs) to disk.", flush=True)


=== Encoding 500 document images on GPU ===
  Encoded 4/500 images in 18.6s...
  Encoded 44/500 images in 205.8s...
  Encoded 84/500 images in 395.1s...
  Encoded 124/500 images in 584.0s...
  Encoded 164/500 images in 773.2s...
  Encoded 204/500 images in 963.2s...
  Encoded 244/500 images in 1151.2s...
  Encoded 284/500 images in 1341.5s...
  Encoded 324/500 images in 1529.8s...
  Encoded 364/500 images in 1718.1s...
  Encoded 404/500 images in 1908.6s...
  Encoded 444/500 images in 2097.6s...
  Encoded 484/500 images in 2288.5s...
  Encoded 500/500 images in 2353.4s...
Completed image encoding of 500 document pages in 2353.4s!
Saved doc embeddings tensor list (500 docs) to disk.


In [6]:
# CELL 6: Live GPU Encoding of 500 Query Texts
import time
import torch
from pathlib import Path

print(f"=== Encoding {len(queries)} human queries on GPU ===", flush=True)
t_start_q = time.time()
batch_size_q = 8
query_embeddings = []

for b_idx in range(0, len(queries), batch_size_q):
    b_queries = queries[b_idx : b_idx + batch_size_q]
    batch_inputs = processor.process_queries(b_queries).to(device)
    with torch.no_grad():
        b_embs = model(**batch_inputs)
        for emb in b_embs:
            query_embeddings.append(emb.to(dtype).cpu())
    if (b_idx // batch_size_q) % 15 == 0 or b_idx + batch_size_q >= len(queries):
        print(f"  Encoded {min(b_idx + batch_size_q, len(queries))}/{len(queries)} queries...", flush=True)

q_elapsed = time.time() - t_start_q
print(f"Completed query encoding of {len(queries)} queries in {q_elapsed:.1f}s!", flush=True)


=== Encoding 500 human queries on GPU ===
  Encoded 8/500 queries...
  Encoded 128/500 queries...
  Encoded 248/500 queries...
  Encoded 368/500 queries...
  Encoded 488/500 queries...
  Encoded 500/500 queries...
Completed query encoding of 500 queries in 21.3s!


In [7]:
# CELL 7: Multi-Vector Late-Interaction MaxSim Scoring & Per-Query Extraction
import math
import numpy as np
import pandas as pd
import torch
from pathlib import Path

print("=== Computing Late-Interaction MaxSim Scores across All Pairs ===", flush=True)
working_dir = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("./working")
working_dir.mkdir(parents=True, exist_ok=True)

# Pre-load doc embeddings to GPU for accelerated scoring if VRAM permits (~150MB total)
try:
    doc_embeddings_gpu = [d.to(device) for d in doc_embeddings]
    print(f"Pre-loaded all {len(doc_embeddings_gpu)} document embeddings onto {device} for fast scoring.", flush=True)
except Exception as e:
    print(f"GPU cache fallback for doc embeddings: {e}", flush=True)
    doc_embeddings_gpu = None

per_query_results = []
rrs = []
hits1 = []
hits5 = []
hits10 = []
ndcgs10 = []

for i in range(len(queries)):
    q_emb = query_embeddings[i].to(device) # (L_q, D)
    target_idx = i # 1-to-1 ground truth correspondence in test_subsampled
    
    # Compute similarity against all documents
    scores = []
    if doc_embeddings_gpu is not None:
        for d_emb in doc_embeddings_gpu:
            sim = torch.matmul(q_emb, d_emb.T) # (L_q, L_d)
            max_sim = torch.max(sim, dim=1).values # (L_q,)
            score = float(torch.sum(max_sim).cpu())
            scores.append(score)
    else:
        chunk_size = 50
        for c_start in range(0, len(doc_embeddings), chunk_size):
            c_docs = [d.to(device) for d in doc_embeddings[c_start : c_start + chunk_size]]
            for d_emb in c_docs:
                sim = torch.matmul(q_emb, d_emb.T)
                max_sim = torch.max(sim, dim=1).values
                score = float(torch.sum(max_sim).cpu())
                scores.append(score)
            
    scores = np.array(scores)
    ranked_indices = np.argsort(-scores)
    target_rank = int(np.where(ranked_indices == target_idx)[0][0]) + 1
    
    rr = 1.0 / target_rank
    h1 = 1.0 if target_rank == 1 else 0.0
    h5 = 1.0 if target_rank <= 5 else 0.0
    h10 = 1.0 if target_rank <= 10 else 0.0
    ndcg10 = (1.0 / math.log2(target_rank + 1)) if target_rank <= 10 else 0.0
    
    rrs.append(rr)
    hits1.append(h1)
    hits5.append(h5)
    hits10.append(h10)
    ndcgs10.append(ndcg10)
    
    top10_fn = [image_filenames[idx] for idx in ranked_indices[:10]]
    
    per_query_results.append({
        "question_id": question_ids[i],
        "doc_id": doc_ids[i],
        "target_image_filename": image_filenames[i],
        "query": queries[i],
        "target_rank": target_rank,
        "reciprocal_rank": round(rr, 6),
        "hit1": int(h1),
        "hit5": int(h5),
        "hit10": int(h10),
        "ndcg10": round(ndcg10, 6),
        "top10_retrieved_filenames": ";".join(top10_fn)
    })
    
    if (i + 1) % 50 == 0 or i + 1 == len(queries):
        print(f"  Scored {i + 1}/{len(queries)} queries | Running MRR: {np.mean(rrs):.4f} | Running Hit@10: {np.mean(hits10):.4f}", flush=True)

df_per_query = pd.DataFrame(per_query_results)
df_per_query.to_csv(working_dir / "colqwen2_docvqa_per_query.csv", index=False)
print(f"\nSaved raw per-query results: {working_dir / 'colqwen2_docvqa_per_query.csv'} ({len(df_per_query)} rows)", flush=True)


=== Computing Late-Interaction MaxSim Scores across All Pairs ===
Pre-loaded all 500 document embeddings onto cuda:0 for fast scoring.
  Scored 50/500 queries | Running MRR: 0.6125 | Running Hit@10: 0.7400
  Scored 100/500 queries | Running MRR: 0.5841 | Running Hit@10: 0.7300
  Scored 150/500 queries | Running MRR: 0.5912 | Running Hit@10: 0.7267
  Scored 200/500 queries | Running MRR: 0.5874 | Running Hit@10: 0.7150
  Scored 250/500 queries | Running MRR: 0.6004 | Running Hit@10: 0.7400
  Scored 300/500 queries | Running MRR: 0.5826 | Running Hit@10: 0.7167
  Scored 350/500 queries | Running MRR: 0.5743 | Running Hit@10: 0.7057
  Scored 400/500 queries | Running MRR: 0.5705 | Running Hit@10: 0.7100
  Scored 450/500 queries | Running MRR: 0.5547 | Running Hit@10: 0.6933
  Scored 500/500 queries | Running MRR: 0.5497 | Running Hit@10: 0.6860

Saved raw per-query results: /kaggle/working/colqwen2_docvqa_per_query.csv (500 rows)


In [8]:
# CELL 8: 10,000 Bootstrap Confidence Interval Resamples
from pathlib import Path
import numpy as np
import pandas as pd

print("=== Computing 10,000 Bootstrap Confidence Intervals ===", flush=True)
working_dir = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("./working")
working_dir.mkdir(parents=True, exist_ok=True)
rng = np.random.default_rng(42)
B = 10000
n = len(rrs)

rr_arr = np.array(rrs)
h1_arr = np.array(hits1)
h5_arr = np.array(hits5)
h10_arr = np.array(hits10)
ndcg_arr = np.array(ndcgs10)

def get_ci(arr):
    boots = [float(np.mean(rng.choice(arr, size=n, replace=True))) for _ in range(B)]
    mean_val = float(np.mean(arr))
    se_val = float(np.std(boots))
    return mean_val, se_val, float(np.percentile(boots, 2.5)), float(np.percentile(boots, 97.5))

mrr_mean, mrr_se, mrr_ci_l, mrr_ci_u = get_ci(rr_arr)
h1_mean, h1_se, h1_ci_l, h1_ci_u = get_ci(h1_arr)
h5_mean, h5_se, h5_ci_l, h5_ci_u = get_ci(h5_arr)
h10_mean, h10_se, h10_ci_l, h10_ci_u = get_ci(h10_arr)
ndcg_mean, ndcg_se, ndcg_ci_l, ndcg_ci_u = get_ci(ndcg_arr)

summary_rows = [{
    "Benchmark": "DocVQA (ViDoRe Subsampled, N=500)",
    "System": "ColQwen2-v1.0 (Live Measured Tesla T4)",
    "N_Queries": n,
    "MRR": round(mrr_mean, 4),
    "MRR_StdError": round(mrr_se, 4),
    "MRR_95%_CI": f"[{mrr_ci_l:.4f}, {mrr_ci_u:.4f}]",
    "Hit@1": round(h1_mean, 4),
    "Hit@5": round(h5_mean, 4),
    "Hit@10": round(h10_mean, 4),
    "nDCG@10": round(ndcg_mean, 4),
    "Mode": "MEASURED_LIVE_GPU"
}]

df_summary = pd.DataFrame(summary_rows)
df_summary.to_csv(working_dir / "colqwen2_docvqa_summary.csv", index=False)

from tabulate import tabulate
print("\n" + tabulate(df_summary, headers="keys", tablefmt="github", index=False))


=== Computing 10,000 Bootstrap Confidence Intervals ===
| Benchmark                          | System                                  |   N_Queries |    MRR |   MRR_StdError | MRR_95%_CI       |   Hit@1 |   Hit@5 |   Hit@10 |   nDCG@10 | Mode              |
|------------------------------------+-----------------------------------------+-------------+--------+----------------+------------------+---------+---------+----------+-----------+-------------------|
| DocVQA (ViDoRe Subsampled, N=500) | ColQwen2-v1.0 (Live Measured Tesla T4) |         500 | 0.5497 |         0.0202 | [0.5106, 0.5892] |   0.482 |   0.622 |    0.686 |    0.5767 | MEASURED_LIVE_GPU |


In [ ]:
# CELL 9: Master Packaging & Summary Diagnostics Export
import zipfile, json
from pathlib import Path
working_dir = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("./working")

audit = {
    "model_id": MODEL_ID,
    "benchmark": "vidore/docvqa_test_subsampled",
    "queries_count": len(queries),
    "images_count": len(images),
    "device": torch.cuda.get_device_name(0),
    "vram_gb": round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2),
    "torch_version": torch.__version__,
    "precision": str(dtype),
    "image_encoding_time_s": round(img_elapsed, 1),
    "query_encoding_time_s": round(q_elapsed, 1),
    "measured_mrr": round(mrr_mean, 4),
    "measured_hit10": round(h10_mean, 4)
}

with open(working_dir / "colqwen2_docvqa_audit.json", "w") as f:
    json.dump(audit, f, indent=2)

zip_path = working_dir / "colqwen2_docvqa_results.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    zf.write(working_dir / "colqwen2_docvqa_per_query.csv", arcname="colqwen2_docvqa_per_query.csv")
    zf.write(working_dir / "colqwen2_docvqa_summary.csv", arcname="colqwen2_docvqa_summary.csv")
    zf.write(working_dir / "colqwen2_docvqa_audit.json", arcname="colqwen2_docvqa_audit.json")

print(f"\nAll DocVQA ColQwen2 evaluation artifacts packaged to: {zip_path}")
print(f"Archive size: {zip_path.stat().st_size / 1024:.1f} KB")
print("\n=== EXECUTION SUCCESSFUL ===", flush=True)
